# CIPHER: Training a Red Team Agent with GRPO

Fine-tunes **LLaMA 3.2-1B** on the CIPHER adversarial network infiltration environment using GRPO.

- **Expected runtime:** ~20-30 min on free T4 GPU
- **Hardware required:** GPU Runtime (Runtime → Change runtime type → T4 GPU)
- **What you'll see:** RED agent reward improving after training

> ⚠️ Make sure GPU is enabled before running!

In [ ]:
# @title Cell 1: Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'NO GPU FOUND — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# @title Cell 2: Install Dependencies (3-5 min)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl>=0.12.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" openenv
!git clone https://github.com/wolfie8935/OPENENV-FINAL cipher_env
%cd cipher_env
!pip install -e . --quiet
print('✅ All dependencies installed')

In [ ]:
# @title Cell 3: Verify Setup
import torch, os
os.environ['LLM_MODE'] = 'stub'

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

from cipher.env_wrapper import CIPHEREnv
env = CIPHEREnv(max_steps=15, llm_mode='stub')
obs, info = env.reset()
_, reward, _, _, step_info = env.step('MOVE toward HVT')
print(f'\n✅ Environment OK | Reward: {reward:+.3f} | Terminal: {step_info["terminal_reason"]}')

In [ ]:
# @title Cell 4: Baseline Measurement (20 episodes)
import numpy as np

env = CIPHEREnv(max_steps=15, llm_mode='stub')
baseline_rewards, baseline_terminals = [], []

for i in range(20):
    obs, info = env.reset()
    _, r, _, _, si = env.step('WAIT')
    baseline_rewards.append(r)
    baseline_terminals.append(si['terminal_reason'])

print(f'Baseline mean reward : {np.mean(baseline_rewards):+.3f}')
print(f'Baseline win rate    : {sum(r>0 for r in baseline_rewards)/20*100:.1f}%')
print(f'Most common terminal : {max(set(baseline_terminals), key=baseline_terminals.count)}')

In [ ]:
# @title Cell 5: Load LLaMA 1B with Unsloth (fits T4 15GB)
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 512  # Keep short for T4 VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Model loaded. Trainable parameters: {params:,} (LoRA only)')

In [ ]:
# @title Cell 6: Build Training Dataset (200 CIPHER scenarios)
import random, json
from datasets import Dataset

SYSTEM_PROMPT = """You are RED PLANNER in CIPHER, an adversarial cyber-security simulation.
Your team infiltrates a 50-node enterprise network across 4 security zones.
Navigate to zone 3 and exfiltrate the target file while staying below BLUE team detection.
Respond with a single JSON object: {"action_type": "...", "target_node": <int|null>, "target_file": <str|null>, "reasoning": "..."}
Valid actions: move, read_file, exfiltrate, wait, abort, plant_false_trail, write_dead_drop"""

rng = random.Random(42)
zone_nodes = {1: list(range(1,16)), 2: list(range(16,31)), 3: list(range(31,46))}
zone_files = {
    1: ['access.log','config.ini','users.txt'],
    2: ['database.conf','api_keys.json','secrets.yml'],
    3: ['classified.pdf','exfil_target.zip','credentials.db']
}

samples = []
for _ in range(200):
    zone = rng.choices([1,2,3], weights=[0.25,0.40,0.35])[0]
    suspicion = rng.choice([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8])
    node = rng.choice(zone_nodes[zone])
    files = rng.sample(zone_files[zone], rng.randint(1,2))

    obs = f"STEP {rng.randint(1,9)} | Zone: {zone} | Node: {node} | Suspicion: {suspicion:.2f} | Files: {', '.join(files)}"

    conv = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': obs},
    ]
    prompt_str = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    samples.append({'prompt': prompt_str, 'zone': zone, 'suspicion': suspicion})

dataset = Dataset.from_list(samples)
print(f'✅ Dataset: {len(dataset)} training scenarios')
print(f'Sample prompt length: ~{len(samples[0]["prompt"])} chars')

In [ ]:
# @title Cell 7: GRPO Reward Function
VALID_ACTIONS = {'move','read_file','exfiltrate','wait','abort','plant_false_trail','write_dead_drop'}

def cipher_reward_fn(completions, **kwargs):
    def _get(key, default, idx):
        v = kwargs.get(key, None)
        if v is None: return default
        return v[idx] if isinstance(v, (list,tuple)) and idx < len(v) else (v[0] if isinstance(v,(list,tuple)) else v)

    rewards = []
    for i, completion in enumerate(completions):
        zone = int(_get('zone', 1, i))
        suspicion = float(_get('suspicion', 0.3, i))
        reward = 0.0

        clean = completion.strip()
        if clean.startswith('```'):
            clean = '\n'.join(l for l in clean.split('\n') if not l.strip().startswith('```')).strip()

        try:
            data = json.loads(clean)
            reward += 0.20
        except:
            rewards.append(-0.20)
            continue

        action = str(data.get('action_type','')).lower().strip()
        reasoning = str(data.get('reasoning',''))

        if action in VALID_ACTIONS: reward += 0.10
        if len(reasoning) > 30: reward += 0.05
        if len(reasoning) > 80: reward += 0.05

        if zone == 3:
            if action == 'exfiltrate': reward += 0.50
            elif action == 'read_file': reward += 0.20
            elif action == 'abort': reward -= 0.30
        elif zone == 2:
            if action == 'move': reward += 0.35
            elif action == 'exfiltrate': reward -= 0.30
        else:
            if action == 'move': reward += 0.35
            elif action == 'exfiltrate': reward -= 0.30

        if suspicion > 0.75:
            if action in ('wait','abort','plant_false_trail'): reward += 0.15
            elif action == 'move': reward -= 0.10

        rewards.append(float(max(-0.5, min(1.0, reward))))
    return rewards

print('✅ Reward function defined')

In [ ]:
# @title Cell 8: GRPO Training (~20-30 min on T4)
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir='./cipher-grpo-output',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=128,
    max_prompt_length=MAX_SEQ_LEN - 128,
    temperature=0.7,
    learning_rate=5e-5,
    bf16=True,
    logging_steps=5,
    save_steps=999999,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=dataset,
    reward_funcs=cipher_reward_fn,
)

print('🚀 Training started — watch reward column trend upward...\n')
train_result = trainer.train()
print(f'\n✅ Training complete!')
if hasattr(train_result, 'metrics'):
    print(f'Final loss: {train_result.metrics.get("train_loss", "n/a")}')

In [ ]:
# @title Cell 9: Evaluate + Generate Improvement Chart
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FastLanguageModel.for_inference(model)
trained_rewards = []
trained_terminals = []

eval_env = CIPHEREnv(max_steps=15, llm_mode='stub')

for i in range(20):
    obs, info = eval_env.reset()
    zone = rng.randint(1,3)
    suspicion = rng.choice([0.1,0.3,0.5,0.7])
    prompt = f"{SYSTEM_PROMPT}\n\nZone: {zone} | Suspicion: {suspicion:.2f} | Choose action:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    action_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)[:200]
    _, r, _, _, si = eval_env.step(action_text)
    trained_rewards.append(r)
    trained_terminals.append(si['terminal_reason'])

# Chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('CIPHER RED Planner — Training Impact', fontsize=13, fontweight='bold')
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#444')

labels = ['Pre-training', 'Post-training']
colors = ['#e74c3c', '#2ecc71']

# Reward comparison
vals = [np.mean(baseline_rewards)*100, np.mean(trained_rewards)*100]
bars = axes[0].bar(labels, vals, color=colors, width=0.5)
axes[0].set_title('Mean Reward', color='white')
axes[0].set_ylabel('Reward (scaled)', color='white')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+0.5, f'{h:.1f}', ha='center', color='white')

# Win rate comparison
wins = [sum(r>0 for r in baseline_rewards)/20*100, sum(r>0 for r in trained_rewards)/20*100]
bars2 = axes[1].bar(labels, wins, color=colors, width=0.5)
axes[1].set_title('Win Rate %', color='white')
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('%', color='white')
for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x()+bar.get_width()/2, h+1.5, f'{h:.1f}%', ha='center', color='white')

delta = np.mean(trained_rewards) - np.mean(baseline_rewards)
fig.text(0.5, 0.01, f'Improvement: {delta:+.3f} reward | Win rate: {wins[0]:.0f}% → {wins[1]:.0f}%',
         ha='center', color='#f1c40f', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('cipher_improvement.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print(f'\n✅ Chart saved: cipher_improvement.png')
print(f'Baseline avg reward  : {np.mean(baseline_rewards):+.3f}')
print(f'Trained avg reward   : {np.mean(trained_rewards):+.3f}')
print(f'Improvement          : {delta:+.3f}')
print(f'Win rate             : {wins[0]:.0f}% → {wins[1]:.0f}%')

In [ ]:
# @title Cell 10: Save Model Locally + Download Results
model.save_pretrained('./cipher-red-planner')
tokenizer.save_pretrained('./cipher-red-planner')
print('✅ Model saved to ./cipher-red-planner/')

# Zip everything for download
import shutil, os
files_to_zip = []
for f in ['cipher_improvement.png', 'rewards_log.csv', 'prompt_evolution_log.jsonl']:
    if os.path.exists(f):
        files_to_zip.append(f)
        print(f'  Found: {f}')

print(f'\n✅ Training complete! Download cipher_improvement.png from the Files panel (left sidebar).')

In [ ]:
# @title Cell 11 (Optional): Download chart directly to your computer
from google.colab import files
files.download('cipher_improvement.png')
print('✅ Download started!')